## Langchain v1.0을 이용해서 챗봇 구현하기

`create_agent()`를 활용해 에이전트를 구현해 보겠습니다.

In [1]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver


import os
from dotenv import load_dotenv

# .env 파일로부터 api_key를 불러오는 코드
load_dotenv()

base_url = "https://mlapi.run/40cc17ae-a89b-4f12-a7d6-13293180fc87/v1"

model_name = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    model=model_name,
    base_url=base_url,
    temperature=0.7,
)

#### 1. 도구 활용하기

그냥 모델은 도구를 사용할 수 없기 때문에 아래와 같은 질문들에 제대로 대답할 수 없습니다.

In [2]:
response = llm.invoke("오늘 서울 날씨는 어때?")
print(response.content)

죄송하지만, 현재 실시간 날씨 정보를 제공할 수는 없습니다. 서울의 날씨를 확인하시려면 기상청 웹사이트나 날씨 앱을 이용해 보시는 것이 좋습니다. 도움이 필요하시면 다른 질문 해주세요!


그렇기 때문에 2가지가 필요합니다.

1. 모델이 사용할 수 있는 도구(Tool)
2. 모델이 도구를 사용할 수 있도록 에이전트로 변환.

#### Tool

Tool이란, LLM이 사용할 수 있는 도구를 말합니다. @Tool 데코레이션을 사용해 구현할 수 있습니다.

tool을 구현할 때는 꼭 함수의 인자 타입과 반환 타입을 명시해 줘야 하고, 함수의 설명을 꼭 작성해 주셔야 합니다.

In [3]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str: # 문자열(str)을 받아서 문자열을 반환한다는 의미.
    """특정 장소의 날씨 정보를 알려주는 기능""" # LLM이 이 설명을 읽고 도구를 사용합니다.
    return f"It's sunny in {location}"

위 코드를 참고해서 오늘의 날짜를 출력하는 함수를 작성해 봅시다.

(오늘의 날짜는 파이썬 기본 라이브러리 `datetime.date.today().isoformat()`를 이용해서 가져올 수 있습니다!)

In [20]:
from asyncio import protocols
from asyncio import protocols
from asyncio import protocols
from asyncio import protocols
from datetime import datetime
from asyncio import protocols
from datetime import date

from langchain.tools import tool

@tool
def get_today_date() -> str:
    """오늘의 날짜를 반환하는 함수"""
    # 4칸 들여쓰기를 해줍니다.
    return date.today().isoformat()


이런 도구들을 사용하기 위해선 llm 모델을 agent로 변환해줘야 합니다.

`create_agent()`를 통해 LLM을 agent로 변환해 줄 수 있습니다!

이 때, `tools`인자를 통해 이 모델이 사용할 수 있는 tool을 제공해 줄 수 있습니다.

In [21]:
# agent로 변환 : 이제 llm이 도구를 스스로 쓸 수 있게 됩니다.
agent = create_agent(
    llm,
    tools=[get_weather, get_today_date], # 모델이 사용할 수 있는 도구
    system_prompt="너는 기상캐스터로써, 사용자가 물어보는 지역에 대한 날씨 정보를 일기예보 형식으로 알려줘야 한다."
)

다시 한 번 날씨를 물어볼까요?

In [23]:
messages = [
    {"role": "user", "content": "오늘은 몇일인가요?"},
]

# agent에 invoke할 때는 {'messages': {'role': 'user', 'content': '안녕하세요'}}와 같은 형식으로 전달
response = agent.invoke(
    {"messages": messages}
)

for message in response['messages']:
    print(message)

content='오늘은 몇일인가요?' additional_kwargs={} response_metadata={} id='3014bd6a-1d79-472f-b9f9-d08f88d0f3f3'
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 104, 'total_tokens': 115, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0a79d5244c', 'id': 'chatcmpl-Djfhias4aZtRWPyaVRDAwKKBaTqwS', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019e6300-ce0e-78f2-b641-6decf7eff116-0' tool_calls=[{'name': 'get_today_date', 'args': {}, 'id': 'call_d1JtdR4BHcis2lcSy0Qx3WX6', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 104, 'output_tokens': 11, 'total_tokens': 115, 'input_token_details': {'audio': 0, 'cache_

모델이 수행한 과정을 좀 더 보기 좋게 출력해 볼게요

In [24]:
print("입력 프롬프트 :", response['messages'][0].content)
print("도구 호출 :", response['messages'][1].tool_calls[0]['name'], response['messages'][1].tool_calls[1]['name'])
print("도구 호출 결과 :", response['messages'][2].content)
print("도구 호출 결과 :", response['messages'][3].content)
print("최종 답변 :\n", response['messages'][4].content)

입력 프롬프트 : 오늘은 몇일인가요?


IndexError: list index out of range

이걸 보면 내부적으로 llm이 아래와 같은 과정으로 답변 생성을 한 것을 알 수 있습니다.

```
[①사용자] 오늘 서울 날씨?
    ↓
[②AI] tool 2개 동시 호출 결정
    ↓
[③tool] get_today_date → '2026-05-13'
[④tool] get_weather   → 'It's sunny in 서울'
    ↓
[⑤AI] 두 결과 합쳐서 최종 답변
```

실습1 : 계산기를 사용하는 LLM

예로부터 LLM이 기초 연산에 약점을 보이는 문제가 많았었는데요,

직접 사칙연산을 수행하는 함수를 구현하고 LLM이 tool로 활용해 연산을 수행할 수 있도록 해봅시다!

`calculator(a: int, b: int, operator: str)->float:` : 2개의 변수 a, b를 입력 받아, operator 연산을 수행한다.

In [26]:
# 계산기 툴 정의하기
from langchain.tools import tool

@tool
def calculator(a: int, b: int, operator: str) -> float:
    """사칙연산(더하기, 빼기, 곱하기, 나누기)을 수행하는 계산기 도구입니다.
    
    a: 첫 번째 숫자
    b: 두 번째 숫자
    operator: 연산자 기호 (+, -, *, /, x, ÷)
    """
    # 1. 더하기 계산
    if operator == '+':
        return float(a + b)
        
    # 2. 빼기 계산
    elif operator == '-':
        return float(a - b)
        
    # 3. 곱하기 계산 (AI가 다양한 곱하기 기호(x, ×, *)를 쓰더라도 처리할 수 있게 합니다)
    elif operator in ('*', 'x', '×'):
        return float(a * b)
        
    # 4. 나누기 계산 (0으로 나누는 에러 방지 포함)
    elif operator in ('/', '÷'):
        if b == 0:
            raise ZeroDivisionError("0으로 나눌 수 없습니다.")
        return float(a / b)
        
    # 5. 그 외의 잘못된 기호가 들어왔을 때
    else:
        raise ValueError(f"지원하지 않는 연산자입니다: {operator}")
# 에이전트 생성
agent = create_agent(
    llm,
    tools=[calculator],
    system_prompt="너는 친절한 계산기야. 사용자가 질문한 사칙연산을 빠르고 정확하게 답변해야 해."
)
# 테스트 질문들 실행
questions = [
    [{"role": "user", "content": "3178 + 254의 계산결과는?"},],
    [{"role": "user", "content": "5137 - 33은?"},],
    [{"role": "user", "content": "223 x 317이 뭐야?"},],
    [{"role": "user", "content": "57을 8로 나누면 뭐야?"},],
]
for question in questions:
    response = agent.invoke(
        {"messages": question}
    )
    print(response['messages'][-1].content)

3178 + 254의 계산 결과는 3432입니다.
5137 - 33은 5104입니다.
223 x 317은 70,691입니다.
57을 8로 나누면 7.125입니다.


앞서 작성했던 날씨, 날짜, 계산기 tool을 모두 하나의 에이전트에 쥐어 준 다음에 모델이 필요할 때마다 tool을 잘 사용하는지 직접 확인해 봅시다!

In [27]:
from langchain_core.messages import AIMessage

agent = create_agent(
    llm,
    tools=[get_today_date, get_weather, calculator],
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": ""}
    ]}
)

for message in response['messages']:
    if isinstance(message, AIMessage) and message.tool_calls:
        for tool_call in message.tool_calls:
            print("tool calling :", tool_call['name'])
    if message.content:
        print("content :", message.content)

content : Hello! How can I assist you today?


#### 시스템 프롬프트 관리

시스템 프롬프트란? : AI의 행동지침을 정의하는 프롬프트.

모델이 지켜야 하는 규칙이나 설정들을 시스템 프롬프트에 담습니다. 에이전트에서는 구체적으로 어떤 규칙들이 들어가는게 좋을까요?

일반적으로 시스템 프롬프트에 들어갈 내용의 형식을 정의해 보자면...

1. 역할 정의
2. 행동 지침
3. 출력 형식
4. 제약 조건

실습 : 이 양식을 지켜서 ai 어시스턴트의 시스템 프롬프트를 다시 작성해 볼까요?

In [29]:
system_prompt = """
여기에 시스템 프롬프트 작성
"""

agent = create_agent(
    llm,
    tools=[calculator, get_weather, get_today_date],
    system_prompt=system_prompt,
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "오늘 부산의 날씨를 알려줘."}
    ]}
)

print(response['messages'][-1].content)

오늘 부산의 날씨는 맑습니다.


이렇듯, 시스템 프롬프트는 길고 자세히 작성하는 것이 좋기 때문에 코드에 넣기 보다는 `agents.md`와 같은 파일로 따로 구성하여 작성을 하게 됩니다.

이는 아래와 같은 이점들 때문입니다.

1. 코드가 깔끔해진다 : 코드 안에 프롬프트가 길게 들어가면 가독성이 떨어짐.
2. 수정이 편하다 : 프롬프트 바꿀 때마다 코드를 볼 필요 없이 `agents.md`만 수정하면 됨.
3. 협업/버전 관리가 쉬움 : 코드가 아닌 프롬프트 변경 이력을 확인하기 용이하고, 개발자가 아닌 사람도 쉽게 편집할 수 있음.

#### 메모리 관리

앞서도 살펴봤듯이, 기본적으로는 이전의 대화 내역을 모두 모델에게 입력하는 식으로 모델의 메모리를 유지합니다.

In [30]:
messages = [
    {"role": "user", "content": "지금부터 네 이름은 트럼프야."},
    {"role": "assistant", "content": "알겠습니다! 지금부터는 저를 트럼프라고 불러주세요! 무엇을 도와드릴까요?"},
    {"role": "user", "content": "트럼프, 나 오늘 주식을 다 잃었어. 어떡하지?"}
]

response = agent.invoke(
    {"messages": messages}
)

print(response['messages'][-1].content)

주식을 잃는 것은 매우 힘든 경험입니다. 다음과 같은 몇 가지 조치를 고려할 수 있습니다:

1. **감정 정리**: 우선 감정을 가라앉히고 상황을 객관적으로 바라보세요.
2. **손실 분석**: 어떤 이유로 손실이 발생했는지 분석해 보세요. 시장의 변동성, 기업의 실적, 개인의 투자 전략 등 다양한 요인이 있을 수 있습니다.
3. **전문가 상담**: 필요하다면 금융 전문가나 투자 상담사와 상담하여 향후 전략을 세워보세요.
4. **장기 투자**: 단기 손실에 연연하지 않고 장기적으로 투자하는 방법을 고려해보세요.
5. **분산 투자**: 리스크를 줄이기 위해 다양한 자산에 분산 투자하는 것이 좋습니다.

힘든 시기지만, 냉정하게 대처하는 것이 중요합니다. 필요한 추가 정보나 조언이 있다면 말씀해 주세요!


하지만 귀찮죠?

직접 대화를 전부 `messages` 리스트에 기록하고, 다시 꺼내오고...

지금은 사용자가 1명이니 충분히 관리할 수 있다곤 해도, 여러 명의 사용자가 생기면 각 사용자마다 대화를 따로 따로 관리해야 하고 이는 굉장히 번거로운 일이 될 겁니다!

`InMemorySaver()`를 사용하면 이를 쉽게 관리할 수 있습니다.

In [32]:
memory = InMemorySaver()

agent = create_agent(
    llm,
    system_prompt="너는 친철한 AI 어시스턴트야",
    checkpointer=memory # checkpointer의 인자로 전달하여 사용.
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "안녕! 내 이름은 민수야. 잘 부탁해!"}]},
    {"configurable": {"thread_id": "1"}} # 사용자 관리를 위한 thread_id를 추가 정보로 제공한다.
)

print(response['messages'][-1].content)

안녕 민수야! 만나서 반가워. 무엇을 도와줄까요?


In [33]:
# 메모리 내용을 출력해 보면 대화 내용들이 모두 `memory` 변수에 잘 저장되어 있는 것을 확인할 수 있습니다.
for message in memory.get(config={"configurable": {"thread_id": "1"}})['channel_values']['messages']:
    print(message.content)

안녕! 내 이름은 민수야. 잘 부탁해!
안녕 민수야! 만나서 반가워. 무엇을 도와줄까요?


In [34]:
# 이제 개발자가 직접 이전 대화 기록들을 입력해 주지 않아도 이전 대화 내용을 기억합니다.
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]},
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

너의 이름은 민수야! 다른 질문이나 도움이 필요하면 언제든지 말해줘.


`thread_id`를 이용해 여러 대화를 각각 관리하는 것도 훨씬 용이합니다!

In [37]:
# thread_id=1 : 민수 / thread_id=2 : 철수

response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 철수야. 오늘 날씨 어때?"}]},
    {"configurable": {"thread_id": "2"}}
)

print(response['messages'][-1].content)

반갑습니다, 철수님! 오늘의 날씨는 지역에 따라 다를 수 있습니다. 현재 위치를 알려주시면 더 정확한 날씨 정보를 제공해 드릴 수 있습니다!


In [38]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]},
    {"configurable": {"thread_id": "2"}}
)

print(response['messages'][-1].content)

철수님이시죠! 도움이 필요하시면 언제든지 말씀해 주세요.


`InMemorySaver()`는 프로그램이 실행되는 동안만 임시로 저장되는 임시 메모리입니다. 따라서 프로그램이 종료되면 사라지게 되죠.

이걸 프로그램이 종료된 뒤에도 기억하게 하기 위해선 해당 정보들을 따로 저장해야 합니다.

실제 서비스에서는 LangGraph의 Store와 DB를 이용해 아래와 같이 데이터를 저장합니다. (실행 X)

In [40]:
from langgraph.checkpoint.sqlite import SqliteSaver

# DB 파일에 체크포인트를 저장 (재시작해도 유지)
with SqliteSaver.from_conn_string("checkpoints.db") as checkpointer:
    graph = builder.compile(checkpointer=checkpointer)

    config = {"configurable": {"thread_id": "user-123"}}
    graph.invoke({"messages": [{"role": "user", "content": "안녕"}]}, config)

    # 같은 thread_id로 다시 호출하면 이전 대화를 이어받음
    graph.invoke({"messages": [{"role": "user", "content": "아까 내가 뭐라고 했지?"}]}, config)


ModuleNotFoundError: No module named 'langgraph.checkpoint.sqlite'

지금 DB까지 연동하고 설정하는건 무리니 다른 방법을 사용해 보겠습니다.

사용자에 대한 정보를 기록할 수 있는 txt 파일을 만든 뒤, tool을 이용해서 해당 파일을 관리할 겁니다!

In [41]:
from langchain_core.runnables import RunnableConfig

@tool
def save_user_info(info: str) -> str:
    """사용자에 대한 정보를 기록합니다. 추후 대화에 도움이 될 만한 사용자 정보를 기록하는데 사용하세요."""
    
    with open(f"ai_memory.txt", "w") as f:
        f.write(info)
    
    return "사용자 정보를 저장했습니다."

@tool
def load_user_info() -> str:
    """저장된 사용자 정보를 불러옵니다."""
    
    try:
        with open(f"ai_memory.txt", "r") as f:
            return f.read()
    except FileNotFoundError:
        return "저장된 정보가 없습니다."

In [42]:
agent = create_agent(
    llm,
    tools=[save_user_info, load_user_info],
    checkpointer=InMemorySaver()
)

In [43]:
response = agent.invoke(
    {
        "messages": {"role": "user", "content": "안녕, 난 민희야. 앞으로 잘 부탁해!"},
    },
    {"configurable": {"thread_id": "1"}}
)

print(response['messages'][-1].content)

안녕하세요, 민희님! 앞으로 잘 부탁드립니다. 궁금한 점이나 도움이 필요하시면 언제든지 말씀해 주세요!


생성된 `ai_memory.txt` 파일을 확인해 봅시다!